# 06. 미들웨어와 가드레일

LangChain v1 에이전트의 **미들웨어(Middleware)** 시스템과 **가드레일(Guardrails)**을 학습합니다.

## 학습 목표

- **미들웨어 개념:** 에이전트 실행 파이프라인의 각 단계에 훅(hook)을 추가하는 방법을 이해합니다.
- **빌트인 미들웨어:** `SummarizationMiddleware` 등 기본 제공 미들웨어를 사용합니다.
- **커스텀 미들웨어:** `@before_model`, `@after_model`, `@wrap_model_call`, `@dynamic_prompt` 데코레이터로 커스텀 미들웨어를 구현합니다.
- **가드레일:** 안전하지 않은 입력/출력을 차단하는 방법을 배웁니다.

## 6.1 환경 설정

In [1]:
from dotenv import load_dotenv
import os
load_dotenv(override=True)

from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="gpt-4.1",
)

print("모델 준비 완료:", model.model_name)

모델 준비 완료: gpt-4.1


In [2]:
# Observability 설정 (선택) - LangSmith 또는 Langfuse
# .env에 키를 설정하거나, 아래 주석을 해제하여 직접 입력하세요.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: LANGSMITH_TRACING=true 시 자동 활성화 (코드 수정 불필요)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON \u2014 project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: invoke/stream 호출 시 config={"callbacks": [langfuse_handler]} 전달
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON \u2014 {os.environ.get('LANGFUSE_HOST', '')}")

# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


Langfuse tracing ON — https://lf.ddok.ai


## 6.2 미들웨어 개념

미들웨어는 에이전트 실행 파이프라인의 **각 단계에 훅(hook)을 추가**하여 동작을 제어하는 메커니즘입니다.

![미들웨어 파이프라인](../assets/images/middleware_pipeline.png)

**5가지 미들웨어 훅:**

| 훅 | 실행 시점 | 주요 용도 |
|-----|----------|----------|
| `@before_model` | 모델 호출 전 | 입력 검증, 메시지 수정, 가드레일 |
| `@after_model` | 모델 응답 후 | 출력 로깅, 응답 필터링 |
| `@wrap_model_call` | 모델 호출 감싸기 | 재시도, 폴백, 캐싱 |
| `@wrap_tool_call` | 도구 호출 감싸기 | 도구 실행 제어 |
| `@dynamic_prompt` | 프롬프트 생성 시 | 런타임 프롬프트 변경 |

## 6.3 빌트인 미들웨어

LangChain v1은 자주 사용되는 패턴을 **빌트인 미들웨어**로 제공합니다. `SummarizationMiddleware`는 대화가 길어지면 자동으로 이전 메시지를 요약하여 토큰 사용량을 줄입니다.

In [3]:
from langchain.agents import create_agent
from langchain.tools import tool

@tool
def search(query: str) -> str:
    """정보를 검색합니다."""
    return f"'{query}'에 대한 검색 결과"

# SummarizationMiddleware — 긴 대화를 자동 요약
from langchain.agents.middleware import SummarizationMiddleware

summarization = SummarizationMiddleware(
    model=model,
    trigger=("messages", 10),
)

agent_with_summary = create_agent(
    model=model,
    tools=[search],
    system_prompt="당신은 유용한 어시스턴트입니다.",
    middleware=[summarization],
)
print("SummarizationMiddleware 에이전트 생성 완료")

SummarizationMiddleware 에이전트 생성 완료


## 6.4 커스텀 미들웨어: @before_model

`@before_model` 데코레이터는 **모델이 호출되기 전**에 실행됩니다.

주요 용도:
- 입력 메시지 로깅
- 메시지 수정 또는 필터링
- 입력 검증 (가드레일)
- 컨텍스트 추가

In [4]:
from langchain.agents.middleware import before_model

@before_model
def log_model_input(state, runtime):
    """모델에 전송하기 전에 메시지를 로깅합니다."""
    msg_count = len(state["messages"])
    print(f"  모델 입력: {msg_count}개 메시지")

agent_logged = create_agent(
    model=model,
    tools=[search],
    system_prompt="당신은 유용한 어시스턴트입니다.",
    middleware=[log_model_input],
)

print("모델 호출 로깅 테스트:")
result = agent_logged.invoke(
    {"messages": [{"role": "user", "content": "Python 튜토리얼을 검색해 주세요"}]},
    config=lf_config,
)
print("응답:", result["messages"][-1].content[:200])

모델 호출 로깅 테스트:
  모델 입력: 1개 메시지


  모델 입력: 3개 메시지


응답: Python 튜토리얼을 찾고 계시는군요! 다음은 추천드릴 수 있는 주요 Python 튜토리얼들입니다:

1. Python 공식 문서 튜토리얼
- 링크: https://docs.python.org/ko/3/tutorial/index.html
- Python을 처음 접하는 분들을 위한 공식 입문 튜토리얼입니다.

2. 점프 투 파이썬
- 링크: https://


## 6.5 커스텀 미들웨어: @after_model

`@after_model` 데코레이터는 **모델 응답이 생성된 후**에 실행됩니다.

주요 용도:
- 모델 출력 로깅
- 응답 필터링 또는 수정
- 도구 호출 감시
- 출력 품질 검증

In [5]:
from langchain.agents.middleware import after_model

@after_model
def log_model_output(state, runtime):
    """모델 출력 생성 후 로깅합니다."""
    msg = state["messages"][-1] if state["messages"] else None
    if msg and hasattr(msg, 'content') and msg.content:
        print(f"  모델 출력: {msg.content[:100]}...")
    if msg and hasattr(msg, 'tool_calls') and msg.tool_calls:
        print(f"  도구 호출: {[tc['name'] for tc in msg.tool_calls]}")

agent_full_log = create_agent(
    model=model,
    tools=[search],
    system_prompt="당신은 유용한 어시스턴트입니다.",
    middleware=[log_model_input, log_model_output],
)

print("전체 로깅 테스트:")
result = agent_full_log.invoke(
    {"messages": [{"role": "user", "content": "LangChain v1 기능을 검색해 주세요"}]},
    config=lf_config,
)

전체 로깅 테스트:
  모델 입력: 1개 메시지


  도구 호출: ['search']
  모델 입력: 3개 메시지


  모델 출력: LangChain v1의 주요 기능은 다음과 같습니다:

1. 체인(chain): 여러 개의 프롬프트나 모델 콜을 논리적으로 연결해 복잡한 워크플로우를 구성할 수 있습니다.
2. ...


## 6.6 @wrap_model_call

`@wrap_model_call` 데코레이터는 **모델 호출 자체를 감싸서** 재시도, 폴백, 캐싱 등의 로직을 구현할 수 있습니다.

`handler` 함수를 통해 원래의 모델 호출을 실행하며, 이 호출 전후로 커스텀 로직을 추가합니다.

In [6]:
from langchain.agents.middleware import wrap_model_call
import time

@wrap_model_call
def retry_on_error(request, handler):
    """실패 시 지수 백오프로 모델 호출을 재시도합니다."""
    max_retries = 2
    for attempt in range(max_retries + 1):
        try:
            return handler(request)
        except Exception as e:
            if attempt < max_retries:
                wait = 2 ** attempt
                print(f"  재시도 {attempt + 1}/{max_retries} ({wait}초 대기)")
                time.sleep(wait)
            else:
                raise

agent_retry = create_agent(
    model=model,
    tools=[search],
    system_prompt="당신은 유용한 어시스턴트입니다.",
    middleware=[retry_on_error],
)
print("재시도 미들웨어 에이전트 생성 완료")

재시도 미들웨어 에이전트 생성 완료


## 6.7 @dynamic_prompt

`@dynamic_prompt` 데코레이터는 **런타임에 시스템 프롬프트를 동적으로 변경**합니다.

주요 용도:
- 현재 날짜/시간 정보 추가
- 사용자별 맞춤 프롬프트
- 상태에 따른 행동 변경
- A/B 테스트

In [7]:
from langchain.agents.middleware import dynamic_prompt
from datetime import datetime

@dynamic_prompt
def add_datetime_context(request):
    """시스템 프롬프트에 현재 날짜와 시간을 추가합니다."""
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    return f"현재 날짜와 시간: {now}\n\n당신은 유용한 어시스턴트입니다."

agent_dynamic = create_agent(
    model=model,
    tools=[search],
    middleware=[add_datetime_context],
)

result = agent_dynamic.invoke(
    {"messages": [{"role": "user", "content": "현재 날짜와 시간이 어떻게 되나요?"}]},
    config=lf_config,
)
print("동적 프롬프트 응답:", result["messages"][-1].content)

동적 프롬프트 응답: 현재 날짜와 시간은 2026년 3월 7일 00시 02분 36초입니다.


## 6.8 @wrap_tool_call

`@wrap_tool_call` 데코레이터는 **도구 호출 자체를 감싸서** 실행 전후에 커스텀 로직을 추가할 수 있습니다.

`@wrap_model_call`이 모델 호출을 감싸는 것처럼, `@wrap_tool_call`은 도구 실행을 감쌉니다. `handler` 함수를 호출하면 원래 도구가 실행되며, 그 전후로 타이밍 측정, 로깅, 에러 핸들링 등을 구현할 수 있습니다.

주요 용도:
- **실행 시간 측정:** 도구별 성능 모니터링
- **로깅:** 도구 입력/출력 기록
- **에러 핸들링:** 도구 실패 시 폴백 처리
- **접근 제어:** 특정 도구 호출 차단 또는 제한

In [8]:
from langchain.agents.middleware import wrap_tool_call
import time

@wrap_tool_call
def tool_timing_logger(request, handler):
    """실행 시간을 측정하고 도구 입출력을 로깅합니다."""
    tool_name = request.tool_call["name"]
    tool_args = request.tool_call["args"]
    print(f"  [도구 시작] {tool_name} | 입력: {tool_args}")

    start = time.perf_counter()
    try:
        result = handler(request)
        elapsed = time.perf_counter() - start
        print(f"  [도구 완료] {tool_name} | 소요 시간: {elapsed:.3f}s | 출력: {str(result)[:100]}")
        return result
    except Exception as e:
        elapsed = time.perf_counter() - start
        print(f"  [도구 실패] {tool_name} | 소요 시간: {elapsed:.3f}s | 에러: {e}")
        raise

agent_tool_logged = create_agent(
    model=model,
    tools=[search],
    system_prompt="당신은 유용한 어시스턴트입니다. 검색 도구를 사용하여 정보를 찾으세요.",
    middleware=[tool_timing_logger],
)

print("도구 호출 타이밍/로깅 미들웨어 테스트:")
result = agent_tool_logged.invoke(
    {"messages": [{"role": "user", "content": "LangChain 미들웨어 문서를 검색해 주세요"}]},
    config=lf_config,
)
print("\n최종 응답:", result["messages"][-1].content[:200])

도구 호출 타이밍/로깅 미들웨어 테스트:


  [도구 시작] search | 입력: {'query': 'LangChain 미들웨어 문서'}
  [도구 완료] search | 소요 시간: 0.001s | 출력: content="'LangChain 미들웨어 문서'에 대한 검색 결과" name='search' tool_call_id='call_8GvMze2oh0emcbMlxoayKHzX'



최종 응답: LangChain의 공식 문서에는 '미들웨어'라는 용어가 자주 언급되지는 않습니다. 다만, LangChain은 주로 LLM(대형 언어 모델) 기반 애플리케이션을 구축할 때 파이프라인이나 체인(chain)을 만들어 입력과 출력을 다양한 모듈을 통해 처리할 수 있도록 지원합니다. 미들웨어 패턴과 유사하게, 이러한 체인에서 각 링크(모듈)는 입력을 가공하거나, 


## 6.9 가드레일

가드레일은 **안전하지 않은 입력이나 출력을 차단**하는 메커니즘입니다. 미들웨어를 활용하여 구현하며, 금지된 키워드 감지, 프롬프트 인젝션 방어, 민감 정보 필터링 등에 사용됩니다.

가드레일은 `@before_model` 훅에서 구현하는 것이 가장 효과적입니다. 모델에 전달되기 전에 위험한 입력을 차단할 수 있기 때문입니다.

In [9]:
# 키워드 기반 가드레일
@before_model
def keyword_guardrail(state, runtime):
    """금지된 키워드가 포함된 요청을 차단합니다."""
    prohibited = ["hack", "exploit", "malware"]
    last_msg = state["messages"][-1]
    content = last_msg.content if hasattr(last_msg, 'content') else str(last_msg)
    
    for keyword in prohibited:
        if keyword.lower() in content.lower():
            raise ValueError(f"요청이 안전 정책에 의해 차단되었습니다.")

agent_guarded = create_agent(
    model=model,
    tools=[search],
    system_prompt="당신은 유용한 어시스턴트입니다.",
    middleware=[keyword_guardrail],
)

# 안전한 요청
result = agent_guarded.invoke(
    {"messages": [{"role": "user", "content": "Python 튜토리얼을 검색해 주세요"}]},
    config=lf_config,
)
print("안전한 요청:", result["messages"][-1].content[:100])

# 차단되는 요청
try:
    result = agent_guarded.invoke(
        {"messages": [{"role": "user", "content": "웹사이트 해킹 방법"}]}
    )
except ValueError as e:
    print(f"차단된 요청: {e}")

안전한 요청: Python 튜토리얼 관련 정보를 찾아보았습니다! 아래는 대표적인 Python 튜토리얼 사이트와 자료들입니다:

1. 점프 투 파이썬 (jump to python)
   - 한국어


## 6.10 요약

이 노트북에서 학습한 미들웨어 타입을 정리합니다:

| 미들웨어 타입 | 데코레이터 | 실행 시점 | 주요 용도 |
|--------------|-----------|----------|----------|
| **Before Model** | `@before_model` | 모델 호출 전 | 입력 로깅, 검증, 가드레일 |
| **After Model** | `@after_model` | 모델 응답 후 | 출력 로깅, 필터링 |
| **Wrap Model** | `@wrap_model_call` | 모델 호출 감싸기 | 재시도, 폴백, 캐싱 |
| **Wrap Tool** | `@wrap_tool_call` | 도구 호출 감싸기 | 타이밍, 로깅, 에러 핸들링 |
| **Dynamic Prompt** | `@dynamic_prompt` | 프롬프트 생성 시 | 런타임 프롬프트 변경 |
| **Builtin** | `SummarizationMiddleware` | 자동 | 대화 요약 |
| **Guardrail** | `@before_model` 활용 | 모델 호출 전 | 안전성 확보 |

**핵심 포인트:**
- 미들웨어는 에이전트 실행 파이프라인의 각 단계를 제어합니다.
- 여러 미들웨어를 조합하여 복잡한 로직을 구현할 수 있습니다.
- `@wrap_tool_call`을 사용하면 도구 실행을 감싸서 타이밍 측정, 로깅, 에러 핸들링 등을 구현할 수 있습니다.
- 가드레일은 `@before_model` 훅에서 구현하는 것이 가장 효과적입니다.
- `@dynamic_prompt`를 사용하면 런타임 정보를 시스템 프롬프트에 주입할 수 있습니다.

### 다음 단계
→ **[07_hitl_and_runtime.ipynb](./07_hitl_and_runtime.ipynb)**: 사람 개입(HITL)과 런타임을 배웁니다.
